# Collect Data

In [105]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [106]:
import pandas as pd
import numpy as np

In [107]:
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [108]:
df = pd.read_csv('final_data_test.csv')
df

,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Nome_Poco_ANP,Latitude,Longitude,Profundidade,Lamina_DAgua,Idade,DIST_VIZINHO_1,DIST_VIZINHO_2,POCOS_NO_RAIO
0,1369.6102,15.4729,8.8017,8.446464e+09,677.0,26.2,2.007566,528.948721,41614.2601,7-AB-103HP-RJS,-22.031677,-39.943771,2623.50,626.0,2721,1.921416,1.948653,2
1,2501.0654,2216.7411,16.6304,1.595928e+10,680.0,26.2,2.007566,528.948721,41614.2601,7-AB-110HP-RJS,-22.062486,-39.948515,2318.70,433.0,3043,0.310285,1.637599,3
2,2609.4715,6033.2146,14.7758,1.417950e+10,695.0,26.2,2.007566,528.948721,41614.2601,7-AB-114H-RJS,-22.063648,-39.951252,2693.30,418.0,2807,0.310285,1.758841,2
3,3093.9543,4336.7473,20.0257,1.921753e+10,643.0,26.2,2.007566,528.948721,41614.2601,7-AB-117H-RJS,-22.048518,-39.938543,2691.20,497.0,2776,1.216240,1.862333,3
4,5975.9191,4934.9738,96.4430,9.255085e+10,735.0,26.2,2.007566,528.948721,41614.2601,7-AB-130HP-RJS,-22.078529,-39.878189,2932.60,962.0,657,0.581175,1.219933,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21130,27293.3776,5.4598,1065.6529,9.260162e+03,545.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-45DA-RJS,-24.599310,-42.546577,5710.40,1913.0,1919,1.532986,1.909536,2
21131,27535.8311,33.0826,1119.4518,9.727656e+03,441.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-18-RJS,-24.682310,-42.470263,5696.21,2011.0,2273,1.467785,1.529022,2
21132,28156.3207,227.0670,1067.9363,9.280003e+03,537.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-30-RJS,-24.651285,-42.439139,5879.69,1987.0,2442,1.087841,1.885437,2
21133,13630.2800,6.8188,860.5805,7.478152e+03,438.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-24D-RJS,-24.668590,-42.471271,5748.20,1956.0,2822,0.886117,1.529022,3


In [109]:
from sklearn.ensemble import IsolationForest

features = df.drop(columns='Nome_Poco_ANP')
features = features.dropna()

iso_forest = IsolationForest(random_state=42)
iso_forest.fit(features)

df = df.loc[features.index].copy()
df['anomaly_score'] = iso_forest.decision_function(features)
df['anomaly'] = iso_forest.predict(features)

df['anomaly'].value_counts()

df

,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Nome_Poco_ANP,Latitude,Longitude,Profundidade,Lamina_DAgua,Idade,DIST_VIZINHO_1,DIST_VIZINHO_2,POCOS_NO_RAIO,anomaly_score,anomaly
0,1369.6102,15.4729,8.8017,8.446464e+09,677.0,26.2,2.007566,528.948721,41614.2601,7-AB-103HP-RJS,-22.031677,-39.943771,2623.50,626.0,2721,1.921416,1.948653,2,0.066775,1
1,2501.0654,2216.7411,16.6304,1.595928e+10,680.0,26.2,2.007566,528.948721,41614.2601,7-AB-110HP-RJS,-22.062486,-39.948515,2318.70,433.0,3043,0.310285,1.637599,3,0.056038,1
2,2609.4715,6033.2146,14.7758,1.417950e+10,695.0,26.2,2.007566,528.948721,41614.2601,7-AB-114H-RJS,-22.063648,-39.951252,2693.30,418.0,2807,0.310285,1.758841,2,0.041593,1
3,3093.9543,4336.7473,20.0257,1.921753e+10,643.0,26.2,2.007566,528.948721,41614.2601,7-AB-117H-RJS,-22.048518,-39.938543,2691.20,497.0,2776,1.216240,1.862333,3,0.067544,1
4,5975.9191,4934.9738,96.4430,9.255085e+10,735.0,26.2,2.007566,528.948721,41614.2601,7-AB-130HP-RJS,-22.078529,-39.878189,2932.60,962.0,657,0.581175,1.219933,4,0.045304,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21130,27293.3776,5.4598,1065.6529,9.260162e+03,545.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-45DA-RJS,-24.599310,-42.546577,5710.40,1913.0,1919,1.532986,1.909536,2,0.040981,1
21131,27535.8311,33.0826,1119.4518,9.727656e+03,441.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-18-RJS,-24.682310,-42.470263,5696.21,2011.0,2273,1.467785,1.529022,2,0.031678,1
21132,28156.3207,227.0670,1067.9363,9.280003e+03,537.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-30-RJS,-24.651285,-42.439139,5879.69,1987.0,2442,1.087841,1.885437,2,0.023739,1
21133,13630.2800,6.8188,860.5805,7.478152e+03,438.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-24D-RJS,-24.668590,-42.471271,5748.20,1956.0,2822,0.886117,1.529022,3,0.047904,1


In [110]:
sample_df = df[df['anomaly'] == 1]
sample_df

,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Nome_Poco_ANP,Latitude,Longitude,Profundidade,Lamina_DAgua,Idade,DIST_VIZINHO_1,DIST_VIZINHO_2,POCOS_NO_RAIO,anomaly_score,anomaly
0,1369.6102,15.4729,8.8017,8.446464e+09,677.0,26.2,2.007566,528.948721,41614.2601,7-AB-103HP-RJS,-22.031677,-39.943771,2623.50,626.0,2721,1.921416,1.948653,2,0.066775,1
1,2501.0654,2216.7411,16.6304,1.595928e+10,680.0,26.2,2.007566,528.948721,41614.2601,7-AB-110HP-RJS,-22.062486,-39.948515,2318.70,433.0,3043,0.310285,1.637599,3,0.056038,1
2,2609.4715,6033.2146,14.7758,1.417950e+10,695.0,26.2,2.007566,528.948721,41614.2601,7-AB-114H-RJS,-22.063648,-39.951252,2693.30,418.0,2807,0.310285,1.758841,2,0.041593,1
3,3093.9543,4336.7473,20.0257,1.921753e+10,643.0,26.2,2.007566,528.948721,41614.2601,7-AB-117H-RJS,-22.048518,-39.938543,2691.20,497.0,2776,1.216240,1.862333,3,0.067544,1
4,5975.9191,4934.9738,96.4430,9.255085e+10,735.0,26.2,2.007566,528.948721,41614.2601,7-AB-130HP-RJS,-22.078529,-39.878189,2932.60,962.0,657,0.581175,1.219933,4,0.045304,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21130,27293.3776,5.4598,1065.6529,9.260162e+03,545.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-45DA-RJS,-24.599310,-42.546577,5710.40,1913.0,1919,1.532986,1.909536,2,0.040981,1
21131,27535.8311,33.0826,1119.4518,9.727656e+03,441.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-18-RJS,-24.682310,-42.470263,5696.21,2011.0,2273,1.467785,1.529022,2,0.031678,1
21132,28156.3207,227.0670,1067.9363,9.280003e+03,537.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-30-RJS,-24.651285,-42.439139,5879.69,1987.0,2442,1.087841,1.885437,2,0.023739,1
21133,13630.2800,6.8188,860.5805,7.478152e+03,438.0,28.7,2.025710,531.127580,38109.9947,7-BUZ-24D-RJS,-24.668590,-42.471271,5748.20,1956.0,2822,0.886117,1.529022,3,0.047904,1


In [111]:
target_column = 'Petroleo'
ignore_columns = [
    'Nome_Poco_ANP', 'Latitude', 'Longitude'
]

In [112]:
# sample_df = df.sample(frac=1, random_state=42)
# sample_df.drop_duplicates(inplace=True)
# columns = [col for col in sample_df.columns if col not in ignore_columns]
# for col in columns:
#     Q1 = sample_df[col].quantile(0.25)
#     Q3 = sample_df[col].quantile(0.75)
#     IQR = Q3 - Q1

#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR

#     sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

# sample_df

In [113]:
sum(sample_df['Nome_Poco_ANP'].value_counts().head(25).values)

np.int64(3195)

In [114]:
X = sample_df.drop([target_column] + ignore_columns, axis=1)
y = sample_df[target_column]

In [115]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=sample_df['Nome_Poco_ANP']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [116]:
# train = sample_df[~sample_df['Nome_Poco_ANP'].isin(pocos_teste)]
# test = sample_df[sample_df['Nome_Poco_ANP'].isin(pocos_teste)]

# X_train = train.drop([target_column] + ignore_columns, axis=1)
# y_train = train[target_column]

# X_test = test.drop([target_column] + ignore_columns, axis=1)
# y_test = test[target_column]

In [117]:
# X = sample_df.drop(ignore_columns + [target_column], axis=1)
# y = sample_df[target_column]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.18, random_state=42)

In [118]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Evaluate Methods

In [119]:
results_df = pd.DataFrame()

def evaluate_and_record(model_name, model, X_train, y_train, X_test, y_test, best_params=None):
    global results_df
    
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    training_time = end_time - start_time
    
    y_pred_treino = model.predict(X_train)
    y_pred_teste = model.predict(X_test)
    
    rmse_treino = np.sqrt(mean_squared_error(y_train, y_pred_treino))
    r2_treino = r2_score(y_train, y_pred_treino)
    mae_treino = mean_absolute_error(y_train, y_pred_treino)
    
    rmse_teste = np.sqrt(mean_squared_error(y_test, y_pred_teste))
    r2_teste = r2_score(y_test, y_pred_teste)
    mae_teste = mean_absolute_error(y_test, y_pred_teste)
    
    new_row = {
        'Modelo': model_name,
        'R2 Treino': r2_treino,
        'R2 Teste': r2_teste,
        'RMSE Treino': rmse_treino,
        'RMSE Teste': rmse_teste,
        'MAE Treino': mae_treino,
        'MAE Teste': mae_teste,
        'Tempo (segundos)': training_time,
        'Hyperparametros': str(best_params) if best_params else 'N/A'
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    return model

In [120]:
def record_gridsearch_details(grid_search_object, model_prefix):
    global results_df
    
    results = pd.DataFrame(grid_search_object.cv_results_)
    
    for index, row in results.iterrows():
        mean_r2_cv = row['mean_test_score']
        
        mean_fit_time = row['mean_fit_time']
        
        model_name_detail = f"{model_prefix} - Combinacao {index + 1}"
        
        new_row = {
            'Modelo': model_name_detail,
            'R2 Teste': mean_r2_cv,
            'RMSE Teste': np.nan,
            'Tempo (s)': mean_fit_time,
            'Detalhes': f"Combin. Testada (Rank: {row['rank_test_score']})",
            'Hyperparametros': str(row['params'])
        }
        
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

# LinearRegression

In [121]:
model = LinearRegression()
evaluate_and_record("Regressao Linear", model, X_train_scaled, y_train, X_test_scaled, y_test)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


# Ridge

In [122]:
alphas = np.logspace(-4, 2, 150)

scoring_metric = 'neg_mean_squared_error'
cv=3
ridge_model = Ridge(random_state=42)
grid_search_ridge = GridSearchCV(
    estimator=ridge_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_ridge.fit(X_train_scaled, y_train)

best_alpha_ridge = grid_search_ridge.best_params_['alpha']
best_ridge_model = grid_search_ridge.best_estimator_

Fitting 3 folds for each of 150 candidates, totalling 450 fits


In [123]:
evaluate_and_record('Ridge Otimo', best_ridge_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_ridge)

,alpha,np.float64(100.0)
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,42


# Lasso

In [124]:
lasso_model = Lasso(random_state=42, max_iter=2000) 
grid_search_lasso = GridSearchCV(
    estimator=lasso_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_lasso.fit(X_train_scaled, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['alpha']
best_lasso_model = grid_search_lasso.best_estimator_

Fitting 3 folds for each of 150 candidates, totalling 450 fits


In [125]:
evaluate_and_record('Lasso Otimo', best_lasso_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_lasso)

,alpha,np.float64(10.803315190764666)
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,2000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,42
,selection,'cyclic'


In [126]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros
0,Regressao Linear,0.741567,0.708473,2089.128326,2370.219360,1297.480049,1464.767425,0.004882,N/A
1,Ridge Otimo,0.741419,0.709967,2089.726958,2364.141321,1295.162418,1462.862849,0.002516,100.0
2,Lasso Otimo,0.741183,0.726875,2090.679003,2294.193928,1288.093385,1449.270648,0.086492,10.803315190764666


# Random Forest

In [22]:
from sklearn.ensemble import RandomForestRegressor

In [23]:
rf_model = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid={
        'n_estimators': [1500, 1550, 1600],
        'min_samples_split': range(1, 10, 2),
        'min_samples_leaf': range(1, 10, 2),
        'bootstrap': [True],
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_rf.fit(X_train_scaled, y_train)

best_n_estimators_rf = grid_search_rf.best_params_['n_estimators']
best_min_samples_split_rf = grid_search_rf.best_params_['min_samples_split']
best_min_samples_leaf_rf = grid_search_rf.best_params_['min_samples_leaf']
best_bootstrap_rf = grid_search_rf.best_params_['bootstrap']
best_rf_model = grid_search_rf.best_estimator_

Fitting 3 folds for each of 75 candidates, totalling 225 fits


c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
45 fits failed out of a total of 225.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
45 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\PC GAMER\D

In [24]:
evaluate_and_record('Random Forest Otimo', best_rf_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_rf.best_params_)

,n_estimators,1500
,criterion,'squared_error'
,max_depth,None
,min_samples_split,3
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Gradient Boosting

In [25]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from xgboost import XGBRegressor

In [27]:
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid={
        'n_estimators': [2000, 2500, 3000],
        'learning_rate': [.0025, .01, .025, .005],
        'max_depth': [9, 11, 13, 15],
        'subsample': [.7, .75, .8],
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)
grid_search_xgb.fit(X_train_scaled, y_train)

best_n_estimators_xgb = grid_search_xgb.best_params_['n_estimators']
best_max_depth_xgb = grid_search_xgb.best_params_['max_depth']
best_learning_rate_xgb = grid_search_xgb.best_params_['learning_rate']
best_subsample_xgb = grid_search_xgb.best_params_['subsample']
best_xgb_model = grid_search_xgb.best_estimator_

Fitting 3 folds for each of 144 candidates, totalling 432 fits


In [28]:
evaluate_and_record('Gradient Boosting Otimo', best_xgb_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_xgb.best_params_)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


# KNN Regression

In [48]:
from sklearn.neighbors import KNeighborsRegressor

In [49]:
knn_model = KNeighborsRegressor()
grid_search_knn = GridSearchCV(
    estimator=knn_model,
    param_grid={
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'p': range(1, 6),
        'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'leaf_size': [30, 40, 50, 60]
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_knn.fit(X_train_scaled, y_train)
best_n_neighbors_knn = grid_search_knn.best_params_['n_neighbors']
best_weights_knn = grid_search_knn.best_params_['weights']
best_p_knn = grid_search_knn.best_params_['p']
best_algorithm_knn = grid_search_knn.best_params_['algorithm']
best_leaf_size_knn = grid_search_knn.best_params_['leaf_size']
best_knn_model = grid_search_knn.best_estimator_

Fitting 3 folds for each of 800 candidates, totalling 2400 fits


In [73]:
evaluate_and_record('KNN Otimo', best_knn_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_knn.best_params_)

,n_neighbors,5
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,1
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [132]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(max_depth=None, min_samples_leaf=2, min_samples_split=4, n_estimators=600, random_state=42)
model.fit(X_train_scaled, y_train)

,n_estimators,600
,criterion,'squared_error'
,max_depth,None
,min_samples_split,4
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [133]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

y_pred = model.predict(X_test_scaled)

print('Random Forest Regressor')
print('RMSE', np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2', r2_score(y_test, y_pred))
print('MAE', mean_absolute_error(y_test, y_pred))

Random Forest Regressor
RMSE 1245.0126061904252
R2 0.9195643508481804
MAE 688.3207269930638


In [129]:
sample_df['Petroleo'].describe()

count    18270.000000
mean      2843.756565
std       4171.036014
min          0.001000
25%        433.160175
50%       1559.944000
75%       3186.595950
max      35197.122600
Name: Petroleo, dtype: float64

In [46]:
from scipy.stats import shapiro

residuos_rf = y_test - model.predict(X_test)
stat, p_valor = shapiro(residuos_rf)
print(f"p-valor: {p_valor}")

p-valor: 4.686519625402188e-32


c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [43]:
import scipy


In [51]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros
0,Regressao Linear,0.838640,0.809746,315.185749,275.772423,148.670916,168.593700,0.002779,N/A
1,Ridge Otimo,0.838627,0.809959,315.198429,275.618106,148.506396,168.508679,0.001711,0.04546928455862406
2,Lasso Otimo,0.838441,0.810358,315.380595,275.328241,148.369102,168.640701,0.141309,0.0001
3,KNN Otimo,1.000000,0.651890,0.000000,373.028183,0.000000,194.960292,0.000746,"{'algorithm': 'auto', 'leaf_size': 30, 'n_neig..."


# SVR (Suport Vector Regression)

In [32]:
from sklearn.svm import SVR

In [33]:
svr_model = SVR()
grid_search_svr = GridSearchCV(
    estimator=svr_model,
    param_grid={
        'C': [11000, 12000, 13000, 14000, 15000],
        'epsilon': [.5, 1, 2, 3],
        'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
        'gamma': ['scale', 'auto'],
        'coef0': [0, .1, .2]
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_svr.fit(X_train_scaled, y_train)

best_C_svr = grid_search_svr.best_params_['C']
best_epsilon_svr = grid_search_svr.best_params_['epsilon']
best_kernel_svr = grid_search_svr.best_params_['kernel']
best_gamma_svr = grid_search_svr.best_params_['gamma']
best_coef0_svr = grid_search_svr.best_params_['coef0']
best_svr_model = grid_search_svr.best_estimator_

Fitting 3 folds for each of 480 candidates, totalling 1440 fits


KeyboardInterrupt: 

In [ ]:
evaluate_and_record('SVR Otimo', best_svr_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_svr.best_params_)

,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,tol,0.001
,C,11000
,epsilon,1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


# GridSearch Details

In [ ]:
record_gridsearch_details(grid_search_ridge, "Ridge GridSearch")
record_gridsearch_details(grid_search_lasso, "Lasso GridSearch")
record_gridsearch_details(grid_search_rf, "Random Forest GridSearch")
record_gridsearch_details(grid_search_xgb, "XGBoost GridSearch")
record_gridsearch_details(grid_search_knn, "KNN GridSearch")
record_gridsearch_details(grid_search_svr, "SVR GridSearch")

In [25]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros
0,Regressao Linear,0.83864,0.809746,315.185749,275.772423,148.670916,168.5937,0.019505,N/A


In [ ]:
results_df.to_csv('model_comparison_results.csv', index=False)